# Analyses of stored results: other measures and selections, across-set tests, the correlation parameter, equivalence, ties

**Computes, without new clustering:** (A) on the imbalanced sets of notebook 29 (sweep at ratio 20 with 3, 5, 8 classes, Ecoli, Dermatology, HCV data), from the partitions kept by 29: IT2 minus each competitor by adjusted mutual information (AMI) at the settings selected by ARI; every method re-selected by AMI and by balanced accuracy and compared in that measure; Holm adjustment over the competitors within each set; the corrected test at the correlation parameter implied by the subsample overlap; on HCV data the recall of every class at the selected settings and two reference partitions. (B) Across-set tests (Wilcoxon, Hodges–Lehmann estimate with exact 95 % interval, sign count) of IT2 against FCM and T1d on the nine real sets with at least five classes and a size ratio of at least 10, on the six sets chosen in advance, and on the 16 main sets, from `results_30.json`; a Friedman test over the ten methods of notebook 29 on the six imbalanced sets. (C) The correlation parameter implied by the overlap of the subsamples. (D) Two one-sided tests of equivalence within ±2 ARI points (margin fixed here, before the run) for the comparisons without a claim on the six sets chosen in advance and on the imbalanced sets. (E) How often the selection from 2–5 % of the labels ties, and the differences with ties broken at random (expected value over the tied settings) instead of by grid order.

**Inputs:** checkpoints `runs29a_*.pkl` of 29, `results_30.json` of 30, `runs24_*.pkl` of 24, `runs32e_parts_*.pkl` of 32.
**Outputs:** `results/33/results_33.json` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, glob
import numpy as np, scipy
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score as _ari, adjusted_mutual_info_score as _ami
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.abspath(os.path.join(RESULTS_DIR, "33")); os.makedirs(os.path.join(OUTDIR, "reports"), exist_ok=True)
BASE = os.path.dirname(OUTDIR)
DIR_29 = os.environ.get("DIR_29", os.path.join(BASE, "29"))
DIR_24 = os.environ.get("DIR_24", os.path.join(BASE, "24"))
DIR_32 = os.environ.get("DIR_32", os.path.join(BASE, "32"))
J30 = os.environ.get("J30", os.path.join(BASE, "30", "results_30.json"))
print(f"python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__}")
for nm, d in (("29", DIR_29), ("24", DIR_24), ("32", DIR_32)):
    print(f"{nm}: {d}  ({len(glob.glob(os.path.join(d, '*.pkl')))} checkpoints)")
print("results_30.json:", J30, os.path.exists(J30))
MM = "IT2FCM-mm"
NAME = {"FCM": "FCM", "T1dec": "T1d", MM: "IT2", "FCMA": "FCMA", "FCMA-d": "FCMA-d", "csiFCM": "csiFCM", "csiFCM-d": "csiFCM-d",
        "MCFCM": "MC-FCM", "RKM": "RKM-IT2FL", "GMM": "GMM"}
REGIME = ["imb_C3_r20", "imb_C5_r20", "imb_C8_r20", "ecoli", "dermatology", "hcv"]
REAL = {"ecoli", "dermatology", "hcv", "zoo", "wine_red", "steel", "ctg", "shuttle", "yeast", "page_blocks", "new_thyroid",
        "iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote"}
FRAC = {"page_blocks": 0.30, "shuttle": 0.05}
RHO_PAPER = {nm: (0.3 / 0.7 if nm == "page_blocks" else 1.0 / 3.0) for nm in REAL}
def rho_overlap(nm):
    """Two stratified subsamples of a fraction f share a fraction f of their points on average."""
    f = FRAC.get(nm, 0.75); return f / (1.0 - f)
FAMS = ("FCM", "T1dec", MM, "FCMA", "FCMA-d", "csiFCM", "csiFCM-d", "MCFCM", "RKM", "GMM")
MARGIN = 0.02

In [ ]:
# Tests and helpers (the claim rule and the corrected test as in the paper).
def tp_nb(d, rho):
    d = np.asarray(d, float); n = len(d)
    if n < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if n else 0.0, t_p=1.0, w_p=1.0, nb_p=1.0, n=n)
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    se = np.sqrt(d.var(ddof=1) * (1.0 / n + rho)); tnb = d.mean() / se if se > 0 else 0.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, nb_p=float(2 * stats.t.sf(abs(tnb), n - 1)), n=n)
def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."
def verdict_nb(a, b):
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."
def tost(d, rho, margin):
    """Two one-sided corrected t-tests of |mean| < margin; returns the larger one-sided p."""
    d = np.asarray(d, float); n = len(d); se = np.sqrt(max(d.var(ddof=1), 1e-30) * (1.0 / n + rho))
    p_lo = stats.t.sf((d.mean() + margin) / se, n - 1); p_hi = stats.t.cdf((d.mean() - margin) / se, n - 1)
    return float(max(p_lo, p_hi))
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); m = len(ps); adj = np.empty(m); run = 0.0
    for i, j in enumerate(o): run = max(run, (m - i) * ps[j]); adj[j] = min(1.0, run)
    return adj
def hodges_lehmann(d, conf=0.95):
    """Median of the Walsh averages and the exact distribution-free interval from the Wilcoxon signed-rank distribution."""
    d = np.asarray(d, float); n = len(d); W = np.sort([(d[i] + d[j]) / 2 for i in range(n) for j in range(i, n)])
    if n < 6: return float(np.median(W)), float("nan"), float("nan")
    counts = np.zeros(n * (n + 1) // 2 + 1); counts[0] = 1
    for r in range(1, n + 1): counts[r:] = counts[r:] + counts[:-r].copy()
    cdf = np.cumsum(counts) / counts.sum(); k = int(np.searchsorted(cdf, (1 - conf) / 2, side="right"))
    k = max(k, 1); return float(np.median(W)), float(W[k - 1]), float(W[len(W) - k])
def balanced_scores(y, lab, C):
    Cp = max(C, int(lab.max()) + 1); M = np.zeros((C, Cp)); np.add.at(M, (y, lab), 1)
    r, c = linear_sum_assignment(-M); rec = np.zeros(C)
    for i, j in zip(r, c): rec[i] = M[i, j] / max(M[i].sum(), 1)
    return rec
def halves(ids):
    ids = sorted(ids); h = len(ids) // 2; return ids[:h], ids[h:]
print("helpers ready")

In [ ]:
# (A) Regime sets: per-setting AMI and balanced accuracy from the partitions kept by 29.
def load29(nm, fam):
    p = os.path.join(DIR_29, f"runs29a_{fam}_{nm}.pkl")
    return pickle.load(open(p, "rb")) if os.path.exists(p) else None
def per_setting(R, fam):
    """{measure: {key: array over instances}} with every measure the mean over the initializations; recall per class kept too."""
    ids = sorted(R); keys = list(R[ids[0]]["rec"][fam]); out = {m: {k: [] for k in keys} for m in ("ari", "ami", "bacc")}; rec = {k: [] for k in keys}
    for s in ids:
        y = R[s]["y"].astype(int); C = R[s]["C"]
        for k in keys:
            labs = [np.asarray(l).astype(int) for l in R[s]["rec"][fam][k]["lab"]]
            out["ari"][k].append(float(np.mean(R[s]["rec"][fam][k]["ari"])))
            out["ami"][k].append(float(np.mean([_ami(y, l) for l in labs])))
            rc = np.mean([balanced_scores(y, l, C) for l in labs], axis=0); rec[k].append(rc); out["bacc"][k].append(float(rc.mean()))
    return ids, {m: {k: np.array(v) for k, v in d.items()} for m, d in out.items()}, rec
def select(vals, A, B):
    """Settings chosen on A and on B by their mean; returns the two choices (each is scored on the other half by the caller)."""
    kA = max(vals, key=lambda k: vals[k][A].mean()); kB = max(vals, key=lambda k: vals[k][B].mean()); return kA, kB
def compare(VI, VZ, sel_meas, score_meas, A, B, rho):
    kIA, kIB = select(VI[sel_meas], A, B); kZA, kZB = select(VZ[sel_meas], A, B)
    a = tp_nb(VI[score_meas][kIA][B] - VZ[score_meas][kZA][B], rho); b = tp_nb(VI[score_meas][kIB][A] - VZ[score_meas][kZB][A], rho)
    return dict(mean=float(np.mean(np.r_[VI[score_meas][kIA][B] - VZ[score_meas][kZA][B], VI[score_meas][kIB][A] - VZ[score_meas][kZB][A]])),
                verdict=verdict(a, b), verdict_nb=verdict_nb(a, b), p_claim=float(max(a["t_p"], a["w_p"], b["t_p"], b["w_p"])),
                nb_p=[a["nb_p"], b["nb_p"]], selected=[str(kIA), str(kZA), str(kIB), str(kZB)],
                diffs=[(VI[score_meas][kIA][B] - VZ[score_meas][kZA][B]).tolist(), (VI[score_meas][kIB][A] - VZ[score_meas][kZB][A]).tolist()])
RES_A = {}; CACHE = {}; t0 = time.time()
for nm in REGIME:
    V = {}
    for fam in FAMS:
        R = load29(nm, fam)
        if R is None: print(f"  {nm} {fam}: checkpoint of 29 not found, skipped"); continue
        ids, vals, rec = per_setting(R, fam); V[fam] = vals; CACHE[(nm, fam)] = (ids, rec, R, vals)
    if MM not in V: continue
    ids = CACHE[(nm, MM)][0]; n = len(ids); h = n // 2; A, B = np.arange(h), np.arange(h, n)
    real = nm in REAL; rp = RHO_PAPER.get(nm, 0.0); ro = rho_overlap(nm) if real else 0.0
    res = dict(n=n, real=real, rho_paper=rp, rho_overlap=ro, compare={})
    for fam in V:
        if fam == MM: continue
        c = {}
        c["ari_by_ari"] = compare(V[MM], V[fam], "ari", "ari", A, B, rp)
        c["ari_by_ari_rho_overlap"] = compare(V[MM], V[fam], "ari", "ari", A, B, ro)
        c["ami_by_ari"] = compare(V[MM], V[fam], "ari", "ami", A, B, rp)
        c["ami_by_ami"] = compare(V[MM], V[fam], "ami", "ami", A, B, rp)
        c["bacc_by_bacc"] = compare(V[MM], V[fam], "bacc", "bacc", A, B, rp)
        for k in c: c[k].pop("diffs") if k != "ari_by_ari" else None
        res["compare"][fam] = c
    fams = [f for f in res["compare"]]
    adj = holm([res["compare"][f]["ari_by_ari"]["p_claim"] for f in fams])
    for f, a in zip(fams, adj): res["compare"][f]["ari_by_ari"]["p_claim_holm"] = float(a)
    res["levels"] = {f: {m: float(np.mean([V[f][m][select(V[f]["ari"], A, B)[0]][B].mean(), V[f][m][select(V[f]["ari"], A, B)[1]][A].mean()])) for m in ("ari", "ami", "bacc")} for f in V}
    RES_A[nm] = res
    print(f"{nm:12s} " + "  ".join(f"{NAME[f]} ARI {100*res['compare'][f]['ari_by_ari']['mean']:+.1f}{'*' if res['compare'][f]['ari_by_ari']['verdict']=='BETTER' else ''} AMI {100*res['compare'][f]['ami_by_ari']['mean']:+.1f}" for f in fams) + f"  ({time.time()-t0:.0f}s)", flush=True)

HCV = {}
if ("hcv", MM) in CACHE:
    ids, _, R, _ = CACHE[("hcv", MM)]; n = len(ids); h = n // 2; A, B = np.arange(h), np.arange(h, n)
    sizes = np.bincount(R[ids[0]]["y"].astype(int)); HCV["class_sizes_instance0"] = sizes.tolist()
    for fam in ("FCM", "T1dec", MM, "GMM", "csiFCM-d"):
        if ("hcv", fam) not in CACHE: continue
        _, rec, Rf, V_ = CACHE[("hcv", fam)]; kA, kB = select(V_["ari"], A, B)
        r = np.mean([rec[kA][i] for i in B] + [rec[kB][i] for i in A], axis=0); HCV[NAME[fam]] = r.tolist()
    ref1, ref2 = [], []
    for s in ids:
        y = R[s]["y"].astype(int); C = R[s]["C"]; big = np.argmax(np.bincount(y)); two = (y != big).astype(int); ref1.append(_ari(y, two))
        rr = np.random.default_rng(33000 + s); vals = []
        for _ in range(20):
            lab = np.where(y == big, 0, rr.integers(1, C, size=len(y))); vals.append(_ari(y, lab))
        ref2.append(float(np.mean(vals)))
    HCV["ari_largest_vs_rest"] = float(np.mean(ref1)); HCV["ari_largest_whole_rest_random"] = float(np.mean(ref2))
    print("HCV recall per class (classes in label order):", {k: np.round(v, 2).tolist() if isinstance(v, list) else v for k, v in HCV.items()})

In [ ]:
# (B) Across-set tests from results_30.json, and a Friedman test over the methods of 29 on the six imbalanced sets.
d30 = json.load(open(J30)) if os.path.exists(J30) else None
GROUPS = {"nine regime real sets": ["ecoli", "yeast", "page_blocks", "zoo", "hcv", "wine_red", "steel", "ctg", "shuttle"],
          "six sets chosen in advance": ["zoo", "hcv", "wine_red", "steel", "ctg", "shuttle"],
          "sixteen main sets": ["equal", "volume", "noise", "density", "volume3", "iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli"]}
RES_B = {}
if d30:
    for g, sets in GROUPS.items():
        RES_B[g] = {}
        for z in ("FCM", "T1dec", "FCMA", "FCMA-d"):
            x = [d30["per_set"][nm]["compare"][f"IT2 vs {z}"]["mean"] for nm in sets if nm in d30["per_set"] and f"IT2 vs {z}" in d30["per_set"][nm]["compare"]]
            if len(x) < len(sets): continue
            x = np.array(x); hl, lo, hi = hodges_lehmann(x)
            RES_B[g][z] = dict(n=len(x), positive=int((x > 0).sum()), median=float(np.median(x)), hl=hl, hl_ci=[lo, hi],
                               wilcoxon_p=float(stats.wilcoxon(x).pvalue), sign_p=float(stats.binomtest(int((x > 0).sum()), int((x != 0).sum())).pvalue))
            print(f"{g:28s} IT2-{NAME[z]:6s} n={len(x)} positive {RES_B[g][z]['positive']}  HL {100*hl:+.1f} [{100*lo:+.1f}, {100*hi:+.1f}]  Wilcoxon p={RES_B[g][z]['wilcoxon_p']:.3f}  sign p={RES_B[g][z]['sign_p']:.3f}")
FR = {}
if RES_A:
    sets = [nm for nm in REGIME if nm in RES_A]; meths = [MM] + [f for f in FAMS if f != MM and all(f in RES_A[nm]["compare"] for nm in sets)]
    M = np.array([[0.0 if f == MM else -RES_A[nm]["compare"][f]["ari_by_ari"]["mean"] for f in meths] for nm in sets])
    ranks = np.array([stats.rankdata(-row) for row in M]); chi = stats.friedmanchisquare(*M.T)
    k, N_ = len(meths), len(sets); se = np.sqrt(k * (k + 1) / (6.0 * N_)); mr = ranks.mean(0)
    pz = {f: float(2 * stats.norm.sf(abs(mr[i] - mr[0]) / se)) for i, f in enumerate(meths) if f != MM}
    ph = dict(zip(pz, holm(list(pz.values()))))
    FR = dict(sets=sets, methods=[NAME[f] for f in meths], chi2=float(chi.statistic), p=float(chi.pvalue), mean_rank={NAME[f]: float(mr[i]) for i, f in enumerate(meths)},
              p_rank={NAME[f]: pz[f] for f in pz}, p_rank_holm={NAME[f]: float(ph[f]) for f in ph})
    print(f"Friedman over {k} methods on the {N_} imbalanced sets: chi2 {FR['chi2']:.2f}, p = {FR['p']:.3f}; mean ranks " + ", ".join(f"{m} {r:.2f}" for m, r in sorted(FR['mean_rank'].items(), key=lambda t: t[1])))

In [ ]:
# (C) The correlation parameter implied by the overlap, and the real-set verdicts of results_30 at that parameter.
RES_C = {"rule": "rho = f/(1-f), f the subsample fraction (overlap of two stratified subsamples)", "per_set": {}}
if d30:
    for nm, r in d30["per_set"].items():
        if not r["real"]: continue
        ro = rho_overlap(nm); key = "3" if abs(ro - 3) < 1e-9 else None
        row = {}
        for k, c in r["compare"].items():
            cr = c["corrected"]
            row[k] = dict(mean=c["mean"], claim=c["verdict"], at_paper=cr["0.333"]["verdict_nb"], at_1=cr["1"]["verdict_nb"], at_3=cr["3"]["verdict_nb"],
                          at_overlap=(cr[key]["verdict_nb"] if key else None))
        RES_C["per_set"][nm] = dict(f=FRAC.get(nm, 0.75), rho_overlap=ro, compare=row)
    for nm, v in RES_C["per_set"].items():
        print(f"{nm:12s} f={v['f']:.2f} rho={v['rho_overlap']:.2f}  " + "  ".join(f"{k[7:]}: claim {c['claim']}, rho=1/3 {c['at_paper']}, rho=3 {c['at_3']}" for k, c in v["compare"].items() if c["claim"] != "n.s."))

In [ ]:
# (D) Equivalence within +-2 ARI points for the comparisons without a claim (six sets: records of 24; imbalanced sets:
# partitions of 29).
RES_D = {}
def inst_vals24(R, f):
    ids = sorted(R); return ids, {k: np.array([np.mean(R[s]["rec"][f][k]["ari"]) for s in ids]) for k in R[ids[0]]["rec"][f]}
for nm in ["zoo", "hcv", "wine_red", "steel", "ctg", "shuttle"]:
    p = os.path.join(DIR_24, f"runs24_{nm}.pkl")
    if not os.path.exists(p): print(f"  {nm}: records of 24 not found"); continue
    R = pickle.load(open(p, "rb")); ids, I = inst_vals24(R, MM); n = len(ids); h = n // 2; A, B = np.arange(h), np.arange(h, n); out = {}
    for z in ("FCM", "T1dec"):
        _, Z = inst_vals24(R, z); kIA, kIB = select(I, A, B); kZA, kZB = select(Z, A, B)
        dB, dA = I[kIA][B] - Z[kZA][B], I[kIB][A] - Z[kZB][A]; a = tp_nb(dB, rho_overlap(nm)); b = tp_nb(dA, rho_overlap(nm))
        out[z] = dict(mean=float(np.mean(np.r_[dB, dA])), claim=verdict(a, b), tost_p=[tost(dB, rho_overlap(nm), MARGIN), tost(dA, rho_overlap(nm), MARGIN)],
                      tost_p_paper_rho=[tost(dB, RHO_PAPER[nm], MARGIN), tost(dA, RHO_PAPER[nm], MARGIN)])
        out[z]["equivalent"] = bool(max(out[z]["tost_p"]) < 0.05); out[z]["equivalent_paper_rho"] = bool(max(out[z]["tost_p_paper_rho"]) < 0.05)
    RES_D[nm] = out
    print(f"{nm:10s} " + "  ".join(f"IT2-{NAME[z]} {100*o['mean']:+.1f} claim {o['claim']} TOST(+-2) p={max(o['tost_p']):.3f} [{'equiv' if o['equivalent'] else 'not shown'}; rho 1/3: {'equiv' if o['equivalent_paper_rho'] else 'not shown'}]" for z, o in out.items()))
for nm, r in RES_A.items():
    out = {}
    for f, c in r["compare"].items():
        if c["ari_by_ari"]["verdict"] != "n.s.": continue
        rho = r["rho_overlap"] if r["real"] else 0.0; dB, dA = map(np.array, c["ari_by_ari"]["diffs"])
        out[NAME[f]] = dict(mean=c["ari_by_ari"]["mean"], tost_p=[tost(dB, rho, MARGIN), tost(dA, rho, MARGIN)]); out[NAME[f]]["equivalent"] = bool(max(out[NAME[f]]["tost_p"]) < 0.05)
    RES_D["regime:" + nm] = out
    if out: print(f"regime {nm:12s} " + "  ".join(f"IT2-{k} {100*o['mean']:+.1f} TOST p={max(o['tost_p']):.3f}{' equiv' if o['equivalent'] else ''}" for k, o in out.items()))
for r in RES_A.values():
    for c in r["compare"].values(): c["ari_by_ari"].pop("diffs", None)

In [ ]:
# (E) Ties in the selection from 2 and 5 % of the labels (partitions of 32), and the differences with ties broken at
# random.
SETS_E = ["imb_C5_r20", "imb_C8_r20", "ecoli", "dermatology", "hcv", "zoo", "shuttle"]; FAMS_E = ("FCM", "T1dec", "FCMA", MM); N_DRAWS = 5
RES_E = {}; t0 = time.time()
for nm in SETS_E:
    p = os.path.join(DIR_32, f"runs32e_parts_{nm}.pkl")
    if not os.path.exists(p): print(f"  {nm}: partitions of 32 not found"); continue
    P = pickle.load(open(p, "rb")); ids = sorted(P); h = len(ids) // 2; res = {}
    for frac in (0.02, 0.05):
        first = {f: {} for f in FAMS_E}; rand = {f: {} for f in FAMS_E}; tie_share = {f: [] for f in FAMS_E}; n_tied = {f: [] for f in FAMS_E}
        for s in ids:
            y = P[s]["y"].astype(int); N = len(y); nL = max(2, int(round(frac * N))); v1 = {f: [] for f in FAMS_E}; v2 = {f: [] for f in FAMS_E}
            for r in range(N_DRAWS):
                L = np.random.default_rng(9000 + 100 * s + r).choice(N, nL, replace=False); U = np.ones(N, bool); U[L] = False
                nq = len(next(iter(P[s]["rec"][FAMS_E[0]].values())))
                for q in range(nq):
                    for f in FAMS_E:
                        keys = list(P[s]["rec"][f]); sl = np.array([_ari(y[L], P[s]["rec"][f][k][q][L]) for k in keys])
                        top = np.flatnonzero(sl >= sl.max() - 1e-12); su = [_ari(y[U], P[s]["rec"][f][keys[i]][q][U]) for i in top]
                        v1[f].append(su[0]); v2[f].append(float(np.mean(su))); tie_share[f].append(len(top) > 1); n_tied[f].append(len(top))
            for f in FAMS_E: first[f][s] = float(np.mean(v1[f])); rand[f][s] = float(np.mean(v2[f]))
        cmp = {}
        for z in ("FCM", "T1dec", "FCMA"):
            for tag, val in (("grid_order", first), ("random", rand)):
                d = np.array([val[MM][s] - val[z][s] for s in ids]); a = tp_nb(d[:h], 0.0); b = tp_nb(d[h:], 0.0)
                cmp[f"IT2 vs {z} ({tag})"] = dict(mean=float(d.mean()), verdict=verdict(a, b))
        res[str(frac)] = dict(tie_share={NAME[f]: float(np.mean(tie_share[f])) for f in FAMS_E}, median_tied={NAME[f]: float(np.median(n_tied[f])) for f in FAMS_E}, compare=cmp)
    RES_E[nm] = res
    print(f"{nm:12s} " + " | ".join(f"{100*float(fr):.0f}%: ties " + ", ".join(f"{k} {100*v:.0f}%" for k, v in r_['tie_share'].items()) + "; IT2-FCM " + f"{100*r_['compare']['IT2 vs FCM (grid_order)']['mean']:+.1f} -> {100*r_['compare']['IT2 vs FCM (random)']['mean']:+.1f}{'*' if r_['compare']['IT2 vs FCM (random)']['verdict']=='BETTER' else ''}" for fr, r_ in res.items()) + f" ({time.time()-t0:.0f}s)", flush=True)

In [ ]:
# Report and result file.
LINES = []
def say(s=""): print(s); LINES.append(s)
star = lambda c: "*" if c["verdict"] == "BETTER" else ("-" if c["verdict"] == "WORSE" else " ")
say(f"NOTEBOOK 33  run {RUN_STAMP}")
say("(A) IMBALANCED SETS, IT2 MINUS EACH (x100; * better / - worse by the claim rule; [corrected test at rho = n_test/n_train | at rho from the overlap]; Holm over competitors)")
for nm, r in RES_A.items():
    say(f"  {nm} (n={r['n']}, rho n_test/n_train {r['rho_paper']:.2f}, rho overlap {r['rho_overlap']:.2f})")
    say("    " + f"{'competitor':10s} {'ARI|ARI':>22s} {'Holm':>6s} {'AMI|ARI':>14s} {'AMI|AMI':>14s} {'bacc|bacc':>14s}")
    for f, c in r["compare"].items():
        a = c["ari_by_ari"]; o = c["ari_by_ari_rho_overlap"]
        say("    " + f"{NAME[f]:10s} {100*a['mean']:+6.1f}{star(a)} [{a['verdict_nb'][:3]}|{o['verdict_nb'][:3]}] {a['p_claim_holm']:6.3f} " + " ".join(f"{100*c[m]['mean']:+7.1f}{star(c[m])} [{c[m]['verdict_nb'][:3]}]" for m in ("ami_by_ari", "ami_by_ami", "bacc_by_bacc")))
    say("    levels (ARI-selected): " + "; ".join(f"{NAME[f]} ARI {100*v['ari']:.1f} AMI {100*v['ami']:.1f} bacc {100*v['bacc']:.1f}" for f, v in r["levels"].items()))
if HCV: say("  HCV data: " + json.dumps({k: (np.round(v, 3).tolist() if isinstance(v, list) else round(v, 4)) for k, v in HCV.items()}))
say("")
say("(B) ACROSS-SET TESTS (per-set mean differences of IT2, label-tuned ARI x100)")
for g, r in RES_B.items():
    for z, v in r.items(): say(f"  {g:28s} IT2-{NAME[z]:6s} n={v['n']:2d} positive {v['positive']:2d}  median {100*v['median']:+.1f}  HL {100*v['hl']:+.1f} [{100*v['hl_ci'][0]:+.1f}, {100*v['hl_ci'][1]:+.1f}]  Wilcoxon p={v['wilcoxon_p']:.4f}  sign p={v['sign_p']:.4f}")
if FR: say(f"  Friedman over {len(FR['methods'])} methods on the six imbalanced sets: chi2 {FR['chi2']:.2f} p={FR['p']:.4f}; mean ranks " + ", ".join(f"{m} {v:.2f}" for m, v in sorted(FR['mean_rank'].items(), key=lambda t: t[1])) + "; Holm p (rank vs IT2) " + ", ".join(f"{m} {v:.3f}" for m, v in FR["p_rank_holm"].items()))
say("")
say("(C) CORRECTED TEST AT THE OVERLAP-IMPLIED rho (= 3 for 75% subsamples): real-set claims")
for nm, v in RES_C["per_set"].items():
    cl = {k: c for k, c in v["compare"].items() if c["claim"] != "n.s."}
    if cl: say(f"  {nm:12s} f={v['f']:.2f} rho={v['rho_overlap']:.2f}  " + "  ".join(f"{k[7:]} {100*c['mean']:+.1f} claim {c['claim']} | rho 1/3 {c['at_paper']} | rho 1 {c['at_1']} | rho 3 {c['at_3']}" for k, c in cl.items()))
say("")
say(f"(D) EQUIVALENCE WITHIN +-{100*MARGIN:.0f} ARI POINTS (two one-sided corrected t-tests on each half; equivalent if both halves p < 0.05)")
for nm, r in RES_D.items():
    if r: say(f"  {nm:20s} " + "  ".join(f"{k}: {100*o['mean']:+.1f} p={max(o['tost_p']):.3f}{' EQUIV' if o['equivalent'] else ''}" for k, o in r.items()))
say("")
say("(E) TIES IN THE SELECTION FROM 2 AND 5% OF THE LABELS; IT2 minus each with ties broken by grid order (as in 32) and at random (expected value)")
for nm, r in RES_E.items():
    for fr, v in r.items():
        say(f"  {nm:12s} {100*float(fr):.0f}%  ties " + ", ".join(f"{k} {100*x:.0f}% (median {v['median_tied'][k]:.0f} tied)" for k, x in v["tie_share"].items()) + "  |  " + "  ".join(f"{k} {100*c['mean']:+.1f}{star(c)}" for k, c in v["compare"].items()))
payload = dict(notebook="33", run_stamp=RUN_STAMP, margin=MARGIN, regime=RES_A, hcv=HCV, across_sets=RES_B, friedman_regime=FR, rho_overlap=RES_C, equivalence=RES_D, ties=RES_E)
for p in (os.path.join(OUTDIR, "results_33.json"), os.path.join(OUTDIR, "results_33_report.txt"), os.path.join(OUTDIR, "reports", f"results_33_report_{RUN_STAMP}.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1, default=str) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n"))
print("saved", os.path.join(OUTDIR, "results_33.json"))